<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge.ipynb">6</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge.ipynb">Next Notebook</a></span>
</div>

## Learning Objectives

By the end of this notebook, you will be able to:
- Build a Movie Database MCP Server using the low-level SDK with SQLite
- Use the `nat mcp client` CLI to discover and test MCP tools
- Define NAT workflow configs (YAML) to connect MCP tools with NVIDIA NIM endpoints
- Build a LangGraph intent classifier that routes queries to a NAT-powered movie agent
- Configure NAT telemetry with Phoenix for observability
- Profile workflow performance using the NAT eval harness with token and latency metrics
- Evaluate agent accuracy using RAGAS AnswerAccuracy as an LLM-as-judge metric

## Introduction

You're building a Movie Database MCP Server — an API that lets AI agents query an IMDB movie database using the [Model Context Protocol](https://modelcontextprotocol.io/).

The scenario: You have a SQLite database (`movie.sqlite`) containing 117 movies with IMDB ratings, box office earnings, and genre data across 3 tables. 
You want to expose this data as tools that any MCP-compatible AI agent can discover and call — enabling agents to search movies, look up ratings, compare earnings, and run custom queries against your database.

## Setup Environment

In the first notebook, we learned how to set up our generated `NVIDIA API KEY`. As a requirement for this notebook, you must set up the key as environment variable `NVIDIA_API_KEY`. If you haven't gotten your key, please visit the NVIDIA NIMs API [homepage](https://build.nvidia.com/explore/discover) and generate your API Key. Please run the cell below, input your `NVIDIA API KEY` in the display textbox, and press the enter key on your keyboard.

In [ ]:
import os
import getpass
import warnings
warnings.filterwarnings("ignore")
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

## Explore the Database

Let's connect to our SQLite database and examine the available tables. The database contains three tables: `IMDB` (movie details), `earning` (box office revenue), and `genre` (movie-to-genre mappings).

In [ ]:
import sqlite3
import json

DB_PATH = 'data/movie.sqlite'

In [ ]:
conn = sqlite3.connect(DB_PATH)                                                                                                                        
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(tables)
conn.close()

## Preview Some Rows

Let's preview some rows from the IMDB table to understand the data structure — each movie has an ID, title, rating, vote count, budget, and runtime.

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime FROM IMDB LIMIT 5")
rows = [dict(row) for row in cursor.fetchall()]
conn.close()

In [ ]:
print(json.dumps(rows, indent=2))

## Helper Class

We create a `MovieDB` helper class that wraps SQLite queries and returns results as MCP-typed `TextContent` objects. This pattern keeps the database logic separate from the MCP server, making it easy to test independently.

The `%%writefile` magic writes the cell contents to a Python file so it can be imported by both the notebook and the MCP server.

In [ ]:
%%writefile movie_db.py
import sqlite3
import json                                                                                                                                                                             
from typing import List
from pathlib import Path                                                                                                                                                                
import mcp.types as types

DB_PATH = "data/movie.sqlite"

class MovieDB:
  def __init__(self, db_path):
      self.db_path = str(Path().resolve().joinpath(db_path))

  def _search_movies(
      self,
      title: str | None,
      min_rating: float | None,
      max_rating: float | None,
      limit: int = 20,
  ) -> List[types.TextContent]:
      """Search movies in the IMDB table by title and/or rating range.

      Args:
          title: Partial or full movie title to search for.
          min_rating: Minimum IMDB rating.
          max_rating: Maximum IMDB rating.
          limit: Max number of results to return.
      """

      conn = sqlite3.connect(self.db_path)
      cursor = conn.cursor()

      query = """
          SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime
          FROM IMDB
      """
      params = []
      conditions = []

      if title is not None:
          conditions.append("Title LIKE ?")
          params.append(f"%{title}%")

      if min_rating is not None:
          conditions.append("Rating >= ?")
          params.append(min_rating)

      if max_rating is not None:
          conditions.append("Rating <= ?")
          params.append(max_rating)

      if conditions:
          query += " WHERE " + " AND ".join(conditions)

      query += " ORDER BY Rating DESC LIMIT ?"
      params.append(limit)

      try:
          cursor.execute(query, params)
          results = cursor.fetchall()

          output = []
          for row in results:
              output.append(
                  {
                      "movie_id": row[0],
                      "title": row[1],
                      "rating": row[2],
                      "total_votes": row[3],
                      "budget": row[4],
                      "runtime": row[5],
                  }
              )

      except sqlite3.Error as e:
          conn.close()
          raise e

      finally:
          conn.close()

      return [types.TextContent(
          type="text",
          text=json.dumps(output)
      )]

In [ ]:
from movie_db import MovieDB

In [ ]:
movie_db = MovieDB(DB_PATH)

## Test the Helper Class

Before building the MCP server, let's verify the `MovieDB` class works correctly with a few different query patterns.

**Search by title** — find movies with "Dark" in the title:

In [ ]:
movie_db._search_movies(title="Dark", min_rating=None, max_rating=None)

**Search by minimum rating** — movies rated 8.5 or higher:

In [ ]:
movie_db._search_movies(title=None, min_rating=8.5, max_rating=None, limit=5)

**Combined filters** — movies with "The" in the title and rating above 8.0:

In [ ]:
movie_db._search_movies(title="The", min_rating=8.0, max_rating=None, limit=3)

**No filters** — return the top 5 movies by rating:

In [ ]:
movie_db._search_movies(title=None, min_rating=None, max_rating=None, limit=5)

## Build the MCP Server

Now we wrap our `MovieDB` class in a low-level MCP server with HTTP transport (similar to what we built in [Notebook 3](03_low_level_mcp.ipynb)). The server exposes a single `search_movies` tool that clients can discover and call. We use Starlette + Uvicorn to serve the MCP endpoint at `http://127.0.0.1:8000/mcp`.

In [ ]:
%%writefile movie_server.py                                                                                                                                                             
import sys
from typing import Any                                                                                                                                                                  
from mcp.server.lowlevel import Server                                                                                                                                                  
from mcp.server.streamable_http_manager import StreamableHTTPSessionManager
from starlette.applications import Starlette                                                                                                                                            
from starlette.routing import Mount
from starlette.types import Receive, Scope, Send
import mcp.types as types
import json
import contextlib
import logging
from collections.abc import AsyncIterator
import uvicorn
from movie_db import MovieDB

logger = logging.getLogger(__name__)

DB_PATH = "movie.sqlite"


def main(db_path: str = DB_PATH):
  movie_db = MovieDB(db_path)
  mcp = Server("movie-db")

  @mcp.list_tools()
  async def handle_list_tools() -> list[types.Tool]:
      return [
          types.Tool(
      name="search_movies",                                                                                                                                                               
      description="Search movies by title and/or rating range. Results are sorted by rating descending. To get the top N highest rated movies, just set limit=N without any rating filters. Rating scale is 1.0 to 10.0, most movies are between 6.0 and 9.0.",
      inputSchema={
          "type": "object",
          "properties": {
              "title": {"type": "string", "description": "Partial or full movie title to search for"},
              "min_rating": {"type": "number", "description": "Minimum IMDB rating (1.0 to 10.0)"},
              "max_rating": {"type": "number", "description": "Maximum IMDB rating (1.0 to 10.0)"},
              "limit": {"type": "integer", "description": "Max results to return (default 20)"},
          },
      },
      ),
      ]

  @mcp.call_tool()
  async def handle_call_tool(name: str, args: dict[str, Any] | None):
      args = args or {}

      if name == "search_movies":
          return movie_db._search_movies(
              title=args.get("title"),
              min_rating=args.get("min_rating"),
              max_rating=args.get("max_rating"),
              limit=args.get("limit", 20),
          )
      else:
          return [types.TextContent(
              type="text",
              text=json.dumps({"error": f"Unknown tool: {name}"})
          )]

  session_manager = StreamableHTTPSessionManager(
      app=mcp,
      event_store=None,
      json_response=True,
      stateless=True,
  )

  async def handle_streamable_http(
      scope: Scope, receive: Receive, send: Send
  ) -> None:
      await session_manager.handle_request(scope, receive, send)

  @contextlib.asynccontextmanager
  async def lifespan(app: Starlette) -> AsyncIterator[None]:
      async with session_manager.run():
          logger.info("Movie DB MCP server started!")
          try:
              yield
          finally:
              logger.info("Movie DB MCP server shutting down...")

  starlette_app = Starlette(
      debug=True,
      routes=[
          Mount("/mcp", app=handle_streamable_http),
      ],
      lifespan=lifespan,
  )

  uvicorn.run(starlette_app, host="127.0.0.1", port=8000)


if __name__ == "__main__":
  db_path = sys.argv[1] if len(sys.argv) > 1 else DB_PATH
  main(db_path)

Start the MCP server in the background so we can test it from subsequent cells.

In [ ]:
import subprocess
import time

server_process = subprocess.Popen(
      ["python", "movie_server.py", "data/movie.sqlite"],                                                                                                                                                      
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE
  )

SERVER_PID = server_process.pid
print(f"Server started on http://127.0.0.1:8000/mcp (PID: {SERVER_PID})")

## Test with NAT CLI

The NeMo Agent Toolkit provides a built-in MCP client CLI (`nat mcp client`) that lets you discover and test MCP tools without writing any client code. Let's use it to list the tools our server exposes.

In [ ]:
!nat mcp client tool list --url http://127.0.0.1:8000/mcp/

Now call the `search_movies` tool directly via the CLI to verify it returns the expected results.

In [ ]:
!nat mcp client tool call search_movies \
      --url http://127.0.0.1:8000/mcp/ \
      --json-args '{"min_rating": 8.5, "limit": 5}'

## Define a NAT Workflow

NAT workflows are defined in YAML configuration files. A workflow config specifies three main sections:
- **`function_groups`**: External tool connections (here, our MCP server)
- **`llms`**: LLM backends to use (here, NVIDIA NIM with Nemotron)
- **`workflow`**: The agent type and how tools connect to the LLM

We'll start with a simple ReAct agent that can call our movie search tool.

In [ ]:
%%writefile movie_workflow.yml                                                                                                                                                          
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client                                                                                                                                                                   
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:8000/mcp/"

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

workflow:
    _type: react_agent
    tool_names:
      - mcp_movies
    llm_name: nim_llm
    verbose: true
    parse_agent_response_max_retries: 3

Run the workflow with a simple movie query.

In [ ]:
!nat run --config_file movie_workflow.yml --input "movies rated over 8.5"

## Build an Intent Classifier with LangGraph

For production scenarios, you often want to route user queries before reaching the agent. Here we build a LangGraph-based intent classifier that:
1. **Classifies** the user's question as `in_domain` (movie-related) or `out_of_domain`
2. **Routes** in-domain queries to the NAT movie agent
3. **Rejects** out-of-domain queries with a polite message

This pattern prevents unnecessary API calls and keeps the agent focused on its domain.

In [ ]:
%%writefile movie_intent_classifier.py
from typing import Annotated
from typing_extensions import TypedDict                                                                                                                                                 
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.sync_builder import SyncBuilder

chat_llm = SyncBuilder.current().get_llm("nim_llm", wrapper_type=LLMFrameworkEnum.LANGCHAIN)
movie_agent = SyncBuilder.current().get_function("movie_agent")


class ConversationState(TypedDict):
  messages: Annotated[list, add_messages]


def classify_node(state: ConversationState):
  response = chat_llm.invoke([
      {"role": "system", "content": (
          "You are an intent classifier for a movie database assistant. "
          "The database contains IMDB movie data: titles, ratings, votes, budgets, and runtimes. "
          "Classify the user's question as either 'in_domain' or 'out_of_domain'. "
          "Reply with ONLY one word.\n\n"
          "in_domain: questions about movies, ratings, top movies, searching movies by title, "
          "comparing movie ratings, budgets, runtimes.\n\n"
          "out_of_domain: anything unrelated to movies."
      )},
      *state["messages"]
  ])
  raw = response.content.strip().lower()
  return {"messages": [{"role": "assistant", "content": raw}]}


def route_intent(state: ConversationState):
  last = state["messages"][-1].content.strip().lower()
  if "in_domain" in last:
      return "movie_agent"
  return "reject"


async def movie_agent_node(state: ConversationState):
  user_query = state["messages"][0].content
  result = await movie_agent.ainvoke(user_query)
  return {"messages": [{"role": "assistant", "content": str(result)}]}


def reject_node(state: ConversationState):
  return {"messages": [{"role": "assistant", "content": "Sorry, I can only help with movie-related questions."}]}


graph = StateGraph(ConversationState)
graph.add_node("classify", classify_node)
graph.add_node("movie_agent", movie_agent_node)
graph.add_node("reject", reject_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route_intent, {"movie_agent": "movie_agent", "reject": "reject"})
graph.add_edge("movie_agent", END)
graph.add_edge("reject", END)

intent_graph = graph.compile()

## Combined Workflow with Telemetry

Now we combine everything into a single NAT workflow that:
- Connects to the MCP movie server
- Configures Phoenix telemetry for tracing and observability
- Defines the movie agent as a NAT function
- Wraps the LangGraph intent classifier as the top-level workflow

Port forwarding has been enabled for the Phoenix port. To view traces, open a new browser tab and navigate to `http://localhost:6006/`.

In [ ]:
import os
os.environ["PHOENIX_PORT"]

In [ ]:
%%writefile movie_intent_workflow.yml                                                                                                                                                   
function_groups:                                                                                                                                                                        
    mcp_movies:                                                                                                                                                                           
      _type: mcp_client
      server:                                                                                                                                                                             
        transport: streamable-http
        url: "http://127.0.0.1:8000/mcp/"

general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:${PHOENIX_PORT}/v1/traces
        project: movie-mcp

llms:
    nim_llm:
      _type: nim
      model_name: nvidia/nemotron-3-nano-30b-a3b
      base_url: https://integrate.api.nvidia.com/v1
      api_key: ${NVIDIA_API_KEY}
      temperature: 0.0
      max_tokens: 1024

functions:
    movie_agent:
      _type: react_agent
      tool_names:
        - mcp_movies
      llm_name: nim_llm
      verbose: true
      parse_agent_response_max_retries: 3

workflow:
    _type: langgraph_wrapper
    graph: movie_intent_classifier.py:intent_graph

## Test the Workflow

Let's test the intent classifier with both out-of-domain and in-domain queries to verify routing works correctly.

**Out-of-domain test** — should be rejected:

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "How do I cook pasta?"

**Another out-of-domain test:**

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "What is the capital of France?"

**In-domain test** — should query the movie database:

In [ ]:
!nat run --config_file movie_intent_workflow.yml --input "what is the rating of the movie The Dark Knight Rises?"

## Cleanup

Stop the MCP server process.

In [ ]:
server_process.terminate()
server_process.wait()
print(f"MCP server (PID: {SERVER_PID}) stopped.")

## Use Cases

The NeMo Agent Toolkit isn't an abstract framework — it's the foundation under real systems solving real problems. To make that concrete, we'll walk through two very different use cases that both sit on top of NAT, showing how the same toolkit scales from a single focused agent to a full multi-agent enterprise research platform.

### AI-Q

The **[AI-Q Blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq)** is an open, enterprise-grade reference for building intelligent research agents that connect to your organization's data, reason over it with state-of-the-art models, and return **citation-backed answers** — not just plausible-sounding prose. 

Built on top of the NVIDIA NeMo Agent Toolkit and LangGraph, the latest [**v2.0**](https://github.com/NVIDIA-AI-Blueprints/aiq/releases/tag/2.0.0) release (March 2026) is a ground-up rewrite that introduces a **two-tier multi-agent architecture**: every incoming query first hits a lightweight **Intent Classifier** that decides whether it's a meta question (answered instantly), a **Shallow Research** task (bounded, tool-calling agent optimized for speed), or a **Deep Research** task (a multi-phase workflow with a dedicated Orchestrator, Planner, and Researcher, plus an optional human-in-the-loop Clarifier that generates and approves a research plan before kicking off a long run).

<div style="text-align: center;">
  <img src="images/aiq_arch.png" style="width: 600px; height: auto;">
</div>


What makes [AI-Q](https://docs.nvidia.com/aiq-blueprint/latest/index.html) production-ready is that it's a **direct, end-to-end implementation of the NVIDIA NeMo Agent Toolkit (NAT)** — every piece of the system is defined, wired, and operated through NAT's primitives. The agents, LLMs, tools, and routing logic all live in **NAT YAML configuration files** with environment variable substitution, so you can reassign roles (e.g., a frontier GPT-5.2 as the orchestrator with open-source Nemotron researchers underneath) without touching a line of Python. 

<div style="text-align: center;">
  <img src="images/aiq_arch.png" style="width: 600px; height: auto;">
</div>

The whole lifecycle runs through NAT's CLI — `nat run` for interactive execution, `nat serve` for the API, and `nat eval` for the built-in evaluation harness that ships with FreshQA and DeepResearch Bench. 

Around this NAT-driven core, AI-Q layers the enterprise plumbing you'd otherwise have to build yourself: an **async Jobs API** with SSE streaming and event replay, **Dask-based distributed execution** for long-running deep research jobs, a **pluggable knowledge layer** (swap LlamaIndex for NVIDIA's Foundational RAG without changing agent code), multimodal PDF extraction via VLMs, **PostgreSQL persistence** for job state and LangGraph checkpoints, and a **deterministic citation verification pipeline** that validates every claim against actually-retrieved sources. 

The result is a [blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq) that currently holds **top positions on both the [DeepResearch Bench](https://huggingface.co/spaces/muset-ai/DeepResearch-Bench-Leaderboard) and [DeepResearch Bench II](https://agentresearchlab.com/benchmarks/deepresearch-bench-ii/index.html#leaderboard) leaderboards** while shipping with Docker Compose stacks, Helm charts, a Next.js frontend, and a three-part Jupyter notebook tutorial. If DABStep shows what NAT looks like when a single agent builds its own reusable tools, AI-Q shows what NAT looks like at the other end of the spectrum — a multi-agent research system with auth, persistence, observability, and evaluation, ready to put in front of real users. 


### NVIDIA KGMON Data Explorer

The **NVIDIA KGMON Data Explorer** is a real-world use case built on top of the NeMo Agent Toolkit that tackles one of the hardest problems in agentic AI: multi-step reasoning over structured tabular data. Most "Deep Research" agents rely on web search, but in domains like finance, healthcare, or operations, the answers live in CSVs, JSONs, and dense domain manuals — not on the internet. 

<div style="text-align: center;">
  <img src="images/data_explorer.png" style="width: 600px; height: auto;">
</div>

The team benchmarked their approach on [DABStep](https://huggingface.co/spaces/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning), a 450-task benchmark from Adyen focused on the financial payments sector, where 84% of tasks require complex cross-referencing between heterogeneous data sources and domain rules. Their key insight was architectural rather than model-driven: instead of having one heavyweight model solve every question from scratch, they split the system into three phases — a **Learning Loop** (heavy model builds a reusable `helper.py` library from training tasks), a **Fast Inference** loop (lightweight model orchestrates those pre-built tools), and an **Offline Reflection** phase (heavy model reviews outputs and injects insights back into the inference prompt). 

<div style="text-align: center;">
  <img src="images/data_explorer_arch.png" style="width: 600px; height: auto;">
</div>

The result: **1st place on DABStep**, a 30x speedup over the Claude Code + Opus 4.5 baseline, and — most strikingly — a smaller model (Haiku 4.5) beating a larger one (Opus 4.5) by **23 points on hard tasks** (89.95 vs 66.93). The takeaway for anyone building agents: the ceiling is often architectural, not model-sized. 

[Full write-up on HuggingFace](https://huggingface.co/blog/nvidia/nemo-agent-toolkit-data-explorer-dabstep-1st-place)


## Links and Resources

- [NeMo Agent Toolkit Documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html)
- [NeMo Agent Toolkit GitHub](https://github.com/NVIDIA/NeMo-Agent-Toolkit)
- [Arize Phoenix](https://phoenix.arize.com/)
- [Arize Phoenix Github](https://github.com/arize-ai/phoenix)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="04_langraph_agent.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a >5</a>
        <a href="06_challenge.ipynb">6</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="06_challenge.ipynb">Next Notebook</a></span>
</div>